In [25]:
# libraries, libraries!
import time
start_time              = time.time()
start_time_r28_download = start_time
print('Importing libraries and setting up paths ...')

from datetime import datetime
import pandas as pd
import os
from pathlib import Path
from tqdm import tqdm, notebook # notebook version of tqdm
#" ...any time you see a loop somewhere in your code in you can simply wrap it in either tdqm() or tqdm_notebook() in Jupyter" 
from utilities import timediff, osprey

# set paths to the driver, urls, and report parameters
eagle_portal = r'https://eagleportal.prescient.co.za'
url_default  = eagle_portal + r'/Default.aspx'
url_r28      = eagle_portal + r'/Queries/Query.aspx?rpt=Reg28withExposure'
pth_py       = r'P:\Investment Operations\GRC\Compliance\Daily\py_reports.xlsm' # variables stored here
pth_dl       = str(Path.home() / 'Downloads')
r28N         = 'Reg 28 Report - Incl Effective Exposure'  # prefix of the file from Eagle

print(f'{timediff(start_time, time.time())} importing libraries and setting up paths', '\n')

Importing libraries and setting up paths ...
0.0sec importing libraries and setting up paths 



In [26]:
# get inputs to pass to Eagle
start_time = time.time()
print('Collecting input data ...')

# get user details, report date, and fund names from the py_report.xlsm sheet
df_downloader = pd.read_excel(pth_py, sheet_name = 'downloader', header = None, usecols = 'D', nrows = 1)
rptDate       = df_downloader.iloc[0,0]     # datetime type
df_funds      = pd.read_excel(pth_py, sheet_name = 'downloader', usecols = 'A').dropna() 
funds         = df_funds['Entity Name'].apply(str.upper)
fund_codes    = df_funds.iloc[:,0].tolist() # list type
fnds_         = (',').join(fund_codes)      # str type

# get credentials
df            = pd.read_excel(pth_py, sheet_name = 'creds', header = None, usecols = 'A', nrows = 2)
aladdin       = df.iloc[0, 0]
sesame        = df.iloc[1, 0]

# check inputs
print(f' Report date: {rptDate.strftime("%A %d %B %Y")}', '\n', f'Portfolios ({len(funds)}): {(", ").join(funds)}')
print(f'{timediff(start_time, time.time())} collecting input data', '\n')

 Report date   : Sunday 30 November 2025 
 Portfolios (141): 3BBCIINC, ABMMBND, ADRRC, ADVMB, AMMARF, AMPBQP, ASBTOS, ASHFLX, BCIFIF, BPROV, CCNPF, CMPFFLEX, CMPFINC, CSIRBQP, ECICBALC, ELCIPF, ENGENIP, ENGENMBF, FEMPBF, GAEMBF, GEMSMEDC, GMRETF, GMRETF2, GRFINV, GTCWP2, HOLADC, HOLDINC, HOLIDC, HOLYPF, HOSMED, IJGCOR, IJGIPF, IMPALA, IMPBAL_C, IMPREF_C, IPIPF, ISPFP, LEZAFFI, LEZALDI, LPIIFTAA, MASAINC, MASAINRF, MASAMMF, MASASI, MASASIRF, MEDINC, MOMBBF, MOMBBRF, MOMFLX, MOMFXB, MOMIPF, MOMPRET, MOMTAAHI, MOMTAALI, MOMTAAMI, MULTICH, MWPFEQU, MWPFILB, MYQIP, NESEQU, NFMWAGG, NGKINC, OMMAIF, PABS, PBNDQ, PBFETF, PCBF, PCCEF, PCEQTF, PCGEARF, PCGEF, PCSHQ, PEQ, PEQF, PETFIP, PEYF, PFFIF, PGCBF, PGCEF, PGIPFA, PGPCEM_C, PGPCGE_C, PGPCZAR, PGPGARF, PGPGBF_C, PGPGIF_C, PGPRF_C, PICPROV, PIF, PIMBAL, PIMEVO, PIPF, PIPFP, PLMED, PLPRNA, PMMF, POIF_C, PORFIP, PPEF, PPOS, PPSBAL, PPSBAL_C, PPSFLEX, PPSNAM, PPSRBNQ, PPWEQU, PRPABF, PRPDBF, PSIF, PSILB, PSPAM, PSSPFEQU, PSTIF, PTIF, QIFFGIF, SA

In [34]:
# determine which files still to be downloaded

# list and then pick out the .csv files from the Downloads folder and then ...
start_time = time.time()
print(f'Identifying the Reg 28 csv files dated {rptDate.strftime("%d %b %Y")} already in the local Downloads folder ...')

# pick out the csv files in the Downloads folder
import re
pattern   = r"^R28I.*\(1\) " + f'{datetime.strftime(rptDate, "%d%b%Y")}' + r"\.csv$"
r28_csvs  = [s for s in os.listdir(pth_dl) if re.match(pattern, s)]

# fund R28I reports still to be downloaded
done = pd.Series(r28_csvs).apply(lambda x: x.replace('R28I ','').replace(f'(1) {rptDate.strftime("%d%b%Y")}.csv',''))
undone = sorted(list(set(funds) ^ set(done)))

print(f'\n {len(funds) - len(r28_csvs)} still to be done, {len(r28_csvs)} {rptDate.strftime("%d%b%Y")} \
R28I csv files in the Downloads folder, {len(funds)} in py_reports.xlsm:','\n','',(', ').join(undone))

print('\n', f'{timediff(start_time, time.time())} identifying the Reg 28 csv files dated {rptDate.strftime("%d %b %Y")} already in \
the local Downloads folder', '\n')

Identifying the Reg 28 csv files dated 30 Nov 2025 already in the local Downloads folder ...

 19 still to be done, 122 30Nov2025 R28I csv files in the Downloads folder, 141 in py_reports.xlsm: 
  SILEQF,SISLP,SMMAIF,SMMIBF,SMMILP,SMMRRF,SNPFEQU,SSLSP,STBFIP,TRFINC,TRFWLTH,UCT3YQ,UCTINC,UCTRFBAL,UCTRFINC,UNISABAL,USAIPF,UWRFCON,UWRFGRO

 0.0sec identifying the Reg 28 csv files dated 30 Nov 2025 already in the local Downloads folder 



In [ ]:
# iterate through the list of reports to be downloaded from Eagle

start_time = time.time()
print(f'Downloading the remaining {len(undone)} fund R28I holdings for Reg 28 reporting as at {rptDate.strftime("%A %d %B %Y")} ...', '\n')

r28_not_downloaded = []
for fund in notebook.tqdm(undone):
    osprey('r28i', fund, rptDate, rptDate, fund, 'csv', aladdin, sesame)

# # download in batches
# from utilities import batch_list
# batches = batch_list(undone, batch_size = 5)
# for index, batch in enumerate(batches):
#     osprey('r28i', batch, rptDate, rptDate, str(index) 'batch' + str(datetime.now().strftime("%Y%m%d%H%M%S")), 'csv', aladdin, sesame)

# open the Downloads folder
os.startfile(os.path.realpath(Path.home() / 'Downloads'))

print('\n\n', f'{timediff(start_time_r28_download, time.time())} downloading the {len(undone)} fund{"s" if len(undone) != 1 else ""} remaining \
fund R28I holdings for Reg 28 reporting as at {rptDate.strftime("%d %B %Y")} ({(time.time() - start_time_r28_download) / (len(funds)):,.1f}sec/fund)')

# C:\Users\hilton.netta\Downloads¶

In [1]:
# !jupyter nbconvert --to script eagle_lt_downloading_csv.ipynb # convert from .ipynb to .py

[NbConvertApp] WARNING | pattern '#' matched no files
[NbConvertApp] WARNING | pattern 'convert' matched no files
[NbConvertApp] WARNING | pattern 'from' matched no files
[NbConvertApp] WARNING | pattern '.ipynb' matched no files
[NbConvertApp] WARNING | pattern 'to' matched no files
[NbConvertApp] WARNING | pattern '.py' matched no files
[NbConvertApp] Converting notebook eagle_lt_downloading_csv.ipynb to script
[NbConvertApp] Writing 4171 bytes to eagle_lt_downloading_csv.py
